# Train on Kaggle — Hierarchical Plant Disease Classification

Template for running the experiments on a Kaggle Notebook (T4 x2 / P100). Steps:

1. Add the dataset `abdallahalidev/plantvillage-dataset` (we use the `color` folder).
2. Clone this repo into `/kaggle/working` (writable) so `src/` imports and the
   scripts can write checkpoints.
3. The persisted split is already in `data/splits/` — reuse it (do **not** regenerate).
4. Train the flat baselines / species head / disease heads, then evaluate the pipeline.

> **Metrics policy:** macro F1 is the **primary** metric (the dataset is imbalanced).
> Accuracy is reported only as a secondary figure — never as the headline result.

## 0. Get the repo into the writable working dir & set up paths

Two delivery modes, tried in order (no edits needed):

1. **Dataset mode (no Internet):** upload this repo as a Kaggle Dataset and add it as
   an Input. The cell finds it under `/kaggle/input/...` (any folder containing `src/`
   + `scripts/`) and copies it to `/kaggle/working` (so scripts can write checkpoints).
2. **Clone mode (needs Internet On):** if no repo dataset is found, `git clone` from
   GitHub. For a private repo add a Kaggle Secret `GITHUB_TOKEN` (Add-ons -> Secrets).

Kaggle's pinned image already has torch / torchvision / timm / albumentations / sklearn,
so **training works fully offline**. (`grad-cam` for Phase-6 error analysis may need
Internet — not required for training.)

In [ ]:
import sys, os, shutil, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/MarijaT99/Neuronske_mreze.git'
REPO_DIR = Path('/kaggle/working/Neuronske_mreze')

def find_repo_in_input():
    """Walk /kaggle/input (depth<=4) for a folder with both src/ and scripts/.
    Skips the 'color' image tree so it stays fast."""
    base = Path('/kaggle/input')
    if not base.exists():
        return None
    stack = [(base, 0)]
    while stack:
        d, depth = stack.pop()
        if (d / 'src').is_dir() and (d / 'scripts').is_dir():
            return d
        if depth < 4:
            for c in d.iterdir():
                if c.is_dir() and c.name != 'color':
                    stack.append((c, depth + 1))
    return None

if not REPO_DIR.exists():
    src = find_repo_in_input()
    if src is not None:
        print('Dataset mode: copying repo from', src)
        shutil.copytree(src, REPO_DIR)
    else:
        print('Clone mode: git clone from GitHub (needs Internet On)')
        clone_url = REPO_URL
        try:
            from kaggle_secrets import UserSecretsClient
            _tok = UserSecretsClient().get_secret('GITHUB_TOKEN')
            if _tok and '@' not in REPO_URL:
                clone_url = REPO_URL.replace('https://', f'https://{_tok}@')
        except Exception:
            pass  # public repo, or no secret set
        subprocess.run(['git', 'clone', '--depth', '1', clone_url, str(REPO_DIR)], check=True)

sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)

import torch
print('repo:', REPO_DIR, '| CUDA:', torch.cuda.is_available(), '|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. Locate the PlantVillage `color` images & sanity-check the split

The exact mount path varies (Kaggle may nest it under
`/kaggle/input/datasets/<user>/<slug>/...`), so we **search** for the `color` folder
instead of hard-coding it. The train scripts read `data.data_root` as a path relative to
the repo root (`data/raw/plantvillage dataset/color`); the next cell symlinks the found
mount there so no config edits are needed.

In [ ]:
def find_color_dir():
    """Find the PlantVillage 'color' dir under /kaggle/input (depth<=5)."""
    stack = [(Path('/kaggle/input'), 0)]
    while stack:
        d, depth = stack.pop()
        if d.name == 'color' and (d / 'Tomato___healthy').is_dir():
            return d
        if depth < 5:
            for c in d.iterdir():
                if c.is_dir():
                    stack.append((c, depth + 1))
    return None

DATA_ROOT = find_color_dir()
assert DATA_ROOT is not None, 'PlantVillage color dir not found in /kaggle/input'
print('DATA_ROOT:', DATA_ROOT)

# The persisted split is versioned in the repo; sanity-check it is present.
from src.data.splits import load_split, LabelMaps
maps = LabelMaps.from_json(REPO_DIR / 'data/splits/label_maps.json')
print('species:', maps.num_species, '| classes:', maps.num_classes)
print('train rows:', len(load_split(REPO_DIR / 'data/splits', 'train')))

## 2. Symlink the dataset into the path the configs expect

In [ ]:
target = REPO_DIR / 'data/raw/plantvillage dataset/color'
target.parent.mkdir(parents=True, exist_ok=True)
if not target.exists():
    os.symlink(DATA_ROOT, target)
print('linked:', target, '->', os.readlink(target) if target.is_symlink() else 'N/A')

## 2b. Offline ImageNet weights for the transfer models

The from-scratch baseline needs no weights, but ResNet-50 / EfficientNet-B0 normally
download ImageNet weights from HuggingFace — which fails with no Internet. Upload the
two `.safetensors` files as a Kaggle Dataset and add it as Input; this cell locates the
folder and points `transfer.py` at it via `PDH_PRETRAINED_DIR` (so timm loads them from
disk). `HF_HUB_OFFLINE=1` skips the network-retry delays. Skip this cell if you only run
the baseline.

In [ ]:
def find_weights_dir():
    """Find a folder containing timm *.safetensors under /kaggle/input."""
    stack = [(Path('/kaggle/input'), 0)]
    while stack:
        d, depth = stack.pop()
        if any(d.glob('*.safetensors')):
            return d
        if depth < 5:
            for c in d.iterdir():
                if c.is_dir() and c.name != 'color':
                    stack.append((c, depth + 1))
    return None

wdir = find_weights_dir()
if wdir is not None:
    os.environ['PDH_PRETRAINED_DIR'] = str(wdir)
    os.environ['HF_HUB_OFFLINE'] = '1'
    print('PDH_PRETRAINED_DIR =', wdir)
    print('weights:', [p.name for p in wdir.glob('*.safetensors')])
else:
    print('WARNING: no *.safetensors found — transfer models will try to download (needs Internet).')

## 3. Flat baselines

The simple CNN (from scratch) and the ResNet-50 flat reference. Each writes
`experiments/<name>/best.pth` (best by val macro F1) + `history.json` + `config.yaml`.

In [ ]:
!python scripts/train_flat.py --config configs/baseline_cnn_flat.yaml --device cuda

In [ ]:
!python scripts/train_flat.py --config configs/resnet50_flat.yaml --device cuda

## 4. Hierarchical: species head + per-species disease heads

In [ ]:
!python scripts/train_species.py --config configs/resnet50_hierarchical.yaml --device cuda

In [ ]:
# Trains the non-trivial species heads in a loop (single-class species are skipped).
!python scripts/train_disease.py --config configs/resnet50_hierarchical.yaml --device cuda

## 5. End-to-end evaluation + flat-vs-hierarchical comparison

Reports end-to-end macro / weighted F1, balanced accuracy, **error propagation**
(what share of end-to-end errors come from a wrong species prediction), and the
flat-vs-hierarchical macro-F1 delta — the key result of the thesis.

In [ ]:
!python scripts/evaluate_pipeline.py \
    --config configs/resnet50_hierarchical.yaml \
    --flat-config configs/resnet50_flat.yaml \
    --device cuda

## 6. Download results

`experiments/<name>/` holds `best.pth`, `history.json`, `config.yaml`,
`test_metrics.json`. Zip the JSON metrics (small) to download; checkpoints are
excluded by default (remove the `-x` flag to include them).

In [ ]:
!cd {REPO_DIR} && zip -r /kaggle/working/experiments.zip experiments -x '*.pth' \
    && echo 'zipped (checkpoints excluded; remove -x to include them)'